# Phase 5: Arm B, Time-Series Foundation Model on 3W

**Model:** Amazon Chronos-2 (120M parameters), used zero-shot: no training on 3W and no labels.
**Score:** forecast surprise per sensor = |actual − median| ÷ 10–90% band width, forecasting 5 minutes ahead from 2 hours of history (computed on a free Colab T4).
**Evaluation:** the same leave-one-well-out scoreboard as Phase 4 (per-well baselines, no-future rule, macro detection vs false alarms per well per day).

**Questions**
1. Can zero-shot Chronos-2 alone (v10a) match the engineered Arm A detectors?
2. Does adding it to Arm A's features (v10b) move the curve up?

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = ROOT / "reports"

curves = pd.concat([
    pd.read_parquet(R / "3w_sweep_v3_vs_v4.parquet").query("features == 'v4'").assign(version="v4"),
    pd.read_parquet(R / "3w_sweep_v6.parquet").assign(version=lambda d: d["config"]),
    pd.read_parquet(R / "3w_sweep_v7.parquet").assign(version="v7_ext_iforest"),
    pd.read_parquet(R / "3w_sweep_v8.parquet").assign(version="v8_ext_ensemble"),
    pd.read_parquet(R / "3w_sweep_v9_ext_lstm_s7.parquet").assign(version="v9_lstm_seed7"),
    pd.read_parquet(R / "3w_sweep_v9_ext_lstm_s123.parquet").assign(version="v9_lstm_seed123"),
    pd.read_parquet(R / "3w_sweep_v10.parquet").assign(version=lambda d: d["config"]),
])

keep = ["v4", "v6_ext", "v7_ext_iforest", "v8_ext_ensemble", "v9_lstm_seed7", "v9_lstm_seed123",
        "v10a_fm_only", "v10b_v4_plus_fm"]
px.line(curves[curves["version"].isin(keep)].sort_values("false_alarms_per_day"),
        x="false_alarms_per_day", y="detection_macro", color="version", markers=True,
        title="Arm A vs Arm B: Chronos-2 zero-shot on the 3W scoreboard").show()

## Results: Chronos-2 zero-shot vs Arm A

Forecast-score coverage: ~32–33% of windows (P-MON-CKP 56%), where two clean hours of history exist.

| Config | q | Macro detection | FA/day | Per-well FA |
|---|---|---|---|---|
| v10a Chronos-2 only | 0.99 | 0.340 | 1.617 | 2.268 |
| v10a | 0.995 | 0.310 | 1.293 | 1.826 |
| v10a | 0.999 | 0.227 | 0.818 | 1.155 |
| v10b Arm A + Chronos-2 | 0.99 | 0.496 | 1.473 | 1.647 |
| v10b | 0.995 | 0.470 | 1.034 | 1.157 |
| v10b | 0.999 | 0.393 | 0.457 | 0.497 |

- Q1: Chronos-2 alone is far below Arm A: matching v4's 0.35 detection costs ~1.6 vs 0.07 false alarms/day.
- Q2: adding it to Arm A's features moves the curve down and right (0.393 at 0.457 FA/day vs ~0.45–0.51 for Arm A at that level).
- Reasons: forecast surprise flags planned operator actions as anomalies (unpredictable ≠ faulty); overlapping forecasts make scores autocorrelated, so the persistence filter barely helps; univariate use ignores how sensors move together.
- Cost: ~1 h of T4 GPU for scoring vs minutes of CPU for Arm A.
- Scope: applies to univariate, zero-shot forecast surprise with 2 h context. Untested: multivariate forecasting with covariates, fine-tuning, combining via the rank-based ensemble.

In [2]:
def detected(file):
    rep = pd.read_parquet(R / file)
    ev = rep[rep["has_event"]].set_index("instance_id")
    return ev["detected"].astype(bool), ev["event_class"]

fm_hit, cls = detected("3w_lowo_v10a_fm_only_q0.99_k5.parquet")
a_hit, _ = detected("3w_lowo_v8_ext_ensemble_q0.99_k5.parquet")

overlap = pd.DataFrame({"class": cls, "fm": fm_hit, "arm_a": a_hit.reindex(fm_hit.index)})
overlap["outcome"] = overlap.apply(
    lambda r: "both" if r.fm and r.arm_a else "only Chronos-2" if r.fm else "only Arm A" if r.arm_a else "neither",
    axis=1)
pd.crosstab(overlap["class"], overlap["outcome"])

outcome,both,neither,only Arm A,only Chronos-2
class,,,,
1,0,0,1,3
2,1,0,21,0
3,31,0,1,0
4,0,277,66,0
5,2,4,4,1
6,0,4,2,0
7,1,29,1,5
8,9,3,1,1
9,2,9,3,0


### Complementarity: which events each detects (v10a Chronos-2 vs v8 ensemble, q=0.99)

| Class | Both | Neither | Only Arm A | Only Chronos-2 |
|---|---|---|---|---|
| 1 BSW | 0 | 0 | 1 | 3 |
| 2 DHSV | 1 | 0 | 21 | 0 |
| 3 Slugging | 31 | 0 | 1 | 0 |
| 4 Instability | 0 | 277 | 66 | 0 |
| 5 Productivity loss | 2 | 4 | 4 | 1 |
| 6 Quick restriction | 0 | 4 | 2 | 0 |
| 7 Scaling | 1 | 29 | 1 | 5 |
| 8 Hydrate (prod.) | 9 | 3 | 1 | 1 |
| 9 Hydrate (service) | 2 | 9 | 3 | 0 |

- Chronos-2 uniquely catches 3 of 4 BSW increases and 5 scaling events (6 vs Arm A's 2), the slow events Arm A missed throughout Phase 4.
- It catches no instability and only 1 DHSV closure because those files (~1.8–1.9 h) are shorter than its 2-hour context: it never scores them.
- Caveats: catches occur at ~1.6 false alarms/day, so some long-event catches may be chance; combining at calibrated thresholds (v10b) made the curve worse.

**Verdict (5.1):** Arm A wins on the scoreboard. Zero-shot Chronos-2 complements it on long, slow events but at a false-alarm cost that cancels the benefit when combined.

**Future work:** shorter context (30–60 min); multivariate forecasting with covariates; chance baseline for long events.